# Is this Hamiltonian file telling the truth

A generator once wrote `n_qubits = n_orbitals` where it should have written `2 * n_orbitals`. Files
describing an 8 electron, 8 orbital active space carried operators half the width the space needs.
Nothing downstream noticed, because nothing downstream checks.

`qbc chem-audit` runs five checks on a file's own declared metadata. The file used here is real: H2
in STO-3G, a 2 electron 2 orbital active space solved to FCI, generated with PySCF and OpenFermion
and shipped in this repository as a test fixture.

In [1]:
import json
from pathlib import Path

from qb_compiler.chem import audit_hamiltonian, load_hamiltonian, measurement_plan

path = Path("../tests/fixtures/hamiltonians/h2_2e2o.json")
raw = json.loads(path.read_text())
meta = raw["metadata"]

print("molecule    :", meta["name"])
print("basis       :", meta["basis"])
print("active space:", meta["active_space"])
print("mapping     :", meta["mapping"])
print("provenance  :", meta["provenance"])
print("reference   :", raw["reference_energy_type"])
print(f"RHF         : {raw['reference_rhf_energy']:.9f} Ha")
print(f"FCI         : {raw['reference_exact_energy']:.9f} Ha")
print(f"correlation : {raw['correlation_energy_mHa']:.4f} mHa")

molecule    : Hydrogen (H2)
basis       : sto-3g
active space: {'n_electrons': 2, 'n_orbitals': 2}
mapping     : jordan_wigner
provenance  : RDKit → PySCF → OpenFermion → JW → sparse_FCI
reference   : FCI_sparse
RHF         : -1.117487425 Ha
FCI         : -1.136654584 Ha
correlation : -19.1672 mHa


In [2]:
hamiltonian = load_hamiltonian(path)
print(audit_hamiltonian(hamiltonian, label=path.name))

ACCEPT: h2_2e2o.json
  [        pass] reference method is correlated
                 reference_energy_type = 'FCI_sparse', names FCI
  [        pass] exact energy sits below RHF
                 exact -1.136654584383908 vs RHF -1.1174874250696742, correlation -0.019167 Ha
  [        pass] provenance chain reaches a correlated solve
                 provenance 'RDKit → PySCF → OpenFermion → JW → sparse_FCI' names FCI and the JW mapping
  [        pass] not flagged synthetic
                 metadata.synthetic = False
  [        pass] qubit count matches the active space
                 n_qubits 4 = 2 x 2 orbitals


## Watch each check fail

A check nobody has watched fail is not evidence that the check works. Each of these damages one
field of an in memory copy. Nothing is written to disk.

In [3]:
import copy


def damaged(**changes):
    record = copy.deepcopy(raw)
    for path_expr, value in changes.items():
        keys = path_expr.split(".")
        target = record
        for key in keys[:-1]:
            target = target[key]
        target[keys[-1]] = value
    return record


cases = {
    "reference energy is the mean field one": damaged(
        reference_exact_energy=raw["reference_rhf_energy"], correlation_energy_mHa=0.0
    ),
    "qubit count is one per spatial orbital": damaged(**{"metadata.n_qubits": 2}),
    "provenance stops before the solve": damaged(
        **{"metadata.provenance": "RDKit -> PySCF -> OpenFermion"}
    ),
    "file flags itself synthetic": damaged(**{"metadata.synthetic": True}),
}

for label, record in cases.items():
    verdict = audit_hamiltonian(record, label=label)
    failed = [c for c in verdict.checks if c.status == "FAIL"]
    print(f"{verdict.verdict:10s} {label}")
    for check in failed:
        print(f"           {check.name}")
        print(f"           {check.detail}")
    print()

REFUSE     reference energy is the mean field one
           exact energy sits below RHF
           exact -1.1174874250696742 vs RHF -1.1174874250696742, correlation +0.000000 Ha; a reference energy identical to the mean field one means no correlation energy is present, which is the signature of a synthetic or placeholder record

REFUSE     qubit count is one per spatial orbital
           qubit count matches the active space
           the operator acts on qubit 3, past the declared n_qubits 2

REFUSE     provenance stops before the solve
           provenance chain reaches a correlated solve
           provenance 'RDKit -> PySCF -> OpenFermion' does not name a correlated solve or a qubit mapping

REFUSE     file flags itself synthetic
           not flagged synthetic
           metadata.synthetic = True: the file says so itself



## Somebody else's file is not a bad file

A file from another group's pipeline usually declares less than ours does. Refusing it would make
this a tool for our own files; passing it would be an assurance nobody measured. So there is a third
verdict, and `--strict` collapses it into a refusal for CI.

In [4]:
sparse = {"n_qubits": 4, "terms": [["Z0 Z1", 0.17], ["X0 X1 Y2 Y3", 0.045], ["I", -0.81]]}

lenient = audit_hamiltonian(sparse, label="a flat term list from elsewhere")
strict = audit_hamiltonian(sparse, label="the same file under --strict", strict=True)

print(lenient.verdict, "->", lenient.undeclared())
print(strict.verdict, "  ->", strict.failures() or "nothing failed, undeclared treated as failure")

INCOMPLETE -> ['reference method is correlated', 'exact energy sits below RHF', 'provenance chain reaches a correlated solve', 'not flagged synthetic', 'qubit count matches the active space']
REFUSE   -> nothing failed, undeclared treated as failure


The reader accepts grouped JSON, flat term lists, OpenFermion operators and Qiskit `SparsePauliOp`,
in both the sparse (`Z0 X3`) and dense (`ZIIX`) spellings.

## What measuring it costs

Separate question, same file. This counts settings and shots. It does not weight by variance and it
makes no claim about the precision of the resulting energy.

In [5]:
plan = measurement_plan(hamiltonian, shots_per_setting=4096)
print(plan)

Measurement plan: ../tests/fixtures/hamiltonians/h2_2e2o.json
  qubits              : 4
  measurable terms    : 14 of 15
  QWC settings        : 5 (grouping factor 2.80x)
  largest group       : 10 terms
  shots per setting   : 4096
  shots, grouped      : 20,480
  shots, term by term : 57,344 (36,864 more than grouped)
  identity offset     : -0.053661
  note                : structural count only: settings and shots follow from which terms share a measurement basis, not from term variance. No shot allocation and no precision claim is implied.


In [6]:
groups = plan.n_settings_qwc
terms = plan.n_measurable_terms
print(f"{terms} measurable terms collapse into {groups} measurement settings")
print(f"identity term carries {plan.identity_coefficient:.6f} Ha and needs no shots")
print(f"grouping saves {plan.shots_term_by_term - plan.shots_qwc:,} shots at this rate")

14 measurable terms collapse into 5 measurement settings
identity term carries -0.053661 Ha and needs no shots
grouping saves 36,864 shots at this rate


H2 is small enough that the numbers are unsurprising. The reason to run this on a real workload is
the spread. Measured across the 14 file corpus this was built against, the setting count spans 1.79
times at 12 qubits (99 to 177, three files) and 1.59 times at 16 (1196 to 1896, nine files), and the
qubit count does not tell you where a given molecule lands. Two jobs of the same width can differ by
nearly a factor of two in what they cost to measure.

The grouping here is textbook greedy qubit wise commuting. Better groupers exist and finding the
minimum is NP hard, so this number is an upper bound on what a good grouper needs. That makes it a
fair basis for a budget and a poor basis for a claim about grouping quality, which is not a claim
this package makes.

In [7]:
import subprocess

done = subprocess.run(
    ["qbc", "chem-audit", str(path), "--strict"], capture_output=True, text=True
)
print(done.stdout.strip()[:400])
print("exit code:", done.returncode, " (0 ACCEPT, 1 INCOMPLETE, 2 REFUSE)")

ACCEPT: ../tests/fixtures/hamiltonians/h2_2e2o.json
  [        pass] reference method is correlated
                 reference_energy_type = 'FCI_sparse', names FCI
  [        pass] exact energy sits below RHF
                 exact -1.136654584383908 vs RHF -1.1174874250696742, correlation -0.019167 Ha
  [        pass] provenance chain reaches a correlated solve
                 provenance 'RDKit
exit code: 0  (0 ACCEPT, 1 INCOMPLETE, 2 REFUSE)
